In [23]:
import cv2
import numpy as np
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
import os
from pathlib import Path


# -----------------------------------------------------------
# Utility functions
# -----------------------------------------------------------

def extract_features(frame, mask):
    """
    Compute:
      - centroid (cx, cy)
      - Hu moments (7 values)
      - color histogram (3-channel histogram)
    """

    # --- Centroid ---
    M = cv2.moments(mask, binaryImage=True)
    if M["m00"] > 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
    else:
        cx, cy = 0, 0

    # --- Hu moments ---
    hu = cv2.HuMoments(M).flatten()
    # Optional for better numeric range:
    hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-12)

    # --- Color histogram (3-channel, 8 bins each) ---
    hist = cv2.calcHist(
        [frame],            # image
        [0, 1, 2],          # channels B,G,R
        mask,               # use only object pixels
        [8, 8, 8],          # histogram binning
        [0, 256, 0, 256, 0, 256]
    ).flatten()

    hist = cv2.normalize(hist, hist).flatten()

    return (cx, cy), hu, hist

def object_distance(features1, features2):
    """Weighted distance for assignment."""
    (c1, hu1, h1) = features1
    (c2, hu2, h2) = features2

    centroid_dist = np.linalg.norm(np.array(c1) - np.array(c2))
    hu_dist = np.linalg.norm(hu1 - hu2)
    hist_dist = cv2.compareHist(h1.astype(np.float32),
                                h2.astype(np.float32),
                                cv2.HISTCMP_BHATTACHARYYA)
    
    # Tune weights depending on your animals
    return 1.0 * centroid_dist + 10.0 * hu_dist + 5.0 * hist_dist


def is_color_valid(frame_bgr, mask, white_vs_straw='lab'):
    """
    Return True if the masked region looks like a chicken (white/neutral or dark),
    False if it looks like straw (yellowish, higher saturation).
    """
    # Ensure mask is binary 0/255
    if mask.dtype != np.uint8:
        mask = mask.astype(np.uint8)
    mask_bin = (mask > 0).astype(np.uint8)

    if white_vs_straw == 'lab':
        lab = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2Lab)
        L, a, b = cv2.split(lab)
        # Only pixels inside the mask
        m = mask_bin.astype(bool)
        if m.sum() == 0:
            return False
        mean_L = float(L[m].mean())
        mean_a = float(a[m].mean())
        mean_b = float(b[m].mean())

        # Heuristic:
        # - Straw tends to have higher b (yellow) and moderate saturation (|a-128|, |b-128|)
        # - White chickens: low chroma (a,b ~ 128), can be bright L, or dark (black chicken: low L)
        chroma = np.hypot(mean_a - 128.0, mean_b - 128.0)

        # Tune thresholds on a short clip (print and inspect)
        # Reject if too yellowish (b >> 128) and chroma high:
        too_yellow = (mean_b - 128.0) > 12.0 and chroma > 14.0

        # Accept if clearly non-yellow OR very low chroma (near white/black)
        accept = (not too_yellow) or (chroma < 10.0)

        return bool(accept)

    else:  # HSV variant
        hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
        H, S, V = cv2.split(hsv)
        m = mask_bin.astype(bool)
        if m.sum() == 0:
            return False
        mean_S = float(S[m].mean())
        # White chickens → low S; straw → higher S (yellowish)
        return mean_S < 60.0  # tune empirically


# -----------------------------------------------------------
# Track class
# -----------------------------------------------------------

class Track:
    """Stores state of a tracked object."""
    next_id = 0
    
    def __init__(self, features, mask):
        self.id = Track.next_id
        Track.next_id += 1
        
        self.features = features
        self.mask = mask
        self.missed = 0    # frames since last detection

In [24]:
# -----------------------------------------------------------
# Main processing
# -----------------------------------------------------------

# VIDEO_PATH = "data/test_30_sec.mp4"
# VIDEO_PATH = "data/test_imgs/%05d.jpg"

image_paths = list(Path("data/test_imgs/").glob("*.jpg"))
files = list(Path("data/test_imgs").glob("*.jpg"))
SAVE_MASKS = True

# cap = cv2.VideoCapture(VIDEO_PATH)
bg = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=120)

tracks = []
frame_idx = 0

# Get total frame count for progress bar
# total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if SAVE_MASKS:
    os.makedirs("data/masks", exist_ok=True)

# tqdm-wrapped frame loop
for image in tqdm(image_paths, desc="Processing frames"):
    cap = cv2.VideoCapture(image)
    ret, frame = cap.read()
    if not ret:
        break
    
    # -------------------------------
    # Step 1 — segmentation
    # -------------------------------
    fg = bg.apply(frame)
    fg = cv2.medianBlur(fg, 5)
    _, fg = cv2.threshold(fg, 200, 255, cv2.THRESH_BINARY)

    kernel = np.ones((5,5), np.uint8)
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, kernel)
    
    contours, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    detected = []   # list of (features, mask)
    
    for cnt in contours:
        if cv2.contourArea(cnt) < 500:
            continue
        
        # mask = np.zeros(frame.shape[:2], dtype=np.uint8)
        # cv2.drawContours(mask, [cnt], -1, 255, -1)

        # features = extract_features(frame, mask)
        # detected.append((features, mask))
    

        mask = np.zeros(frame.shape[:2], dtype=np.uint8)
        cv2.drawContours(mask, [cnt], -1, 255, -1)

        # Reject straw-like blobs
        if not is_color_valid(frame, mask, white_vs_straw='lab'):
            continue

        features = extract_features(frame, mask)
        detected.append((features, mask))

    # -------------------------------
    # Step 2 — Data Association
    # -------------------------------
    if len(tracks) == 0:
        for features, mask in detected:
            tracks.append(Track(features, mask))
    else:
        cost = np.zeros((len(tracks), len(detected)))
        for i, tr in enumerate(tracks):
            for j, (f, m) in enumerate(detected):
                cost[i, j] = object_distance(tr.features, f)
        
        row_ind, col_ind = linear_sum_assignment(cost)

        assigned_tracks = set()
        assigned_dets = set()

        for r, c in zip(row_ind, col_ind):
            if cost[r, c] < 200:
                tracks[r].features = detected[c][0]
                tracks[r].mask = detected[c][1]
                tracks[r].missed = 0
                assigned_tracks.add(r)
                assigned_dets.add(c)

        for i, tr in enumerate(tracks):
            if i not in assigned_tracks:
                tr.missed += 1

        tracks = [t for t in tracks if t.missed < 10]

        for j, (f, m) in enumerate(detected):
            if j not in assigned_dets:
                tracks.append(Track(f, m))
    
    # -------------------------------
    # Step 3 — Save masks
    # -------------------------------
    if SAVE_MASKS:
        for tr in tracks:
            filename = f"data/masks/frame_{frame_idx:05d}_ID_{tr.id}.png"
            cv2.imwrite(filename, tr.mask)
    
    # -------------------------------
    # Step 4 — visualization (optional)
    # -------------------------------
    vis = frame.copy()
    for tr in tracks:
        cx, cy = tr.features[0]
        cv2.putText(vis, f"ID {tr.id}", (cx, cy),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
        cv2.circle(vis, (cx, cy), 5, (0,255,0), -1)
    
    cv2.imshow("tracking", vis)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    
    frame_idx += 1

cap.release()
cv2.destroyAllWindows()


Processing frames: 100%|██████████| 678/678 [00:16<00:00, 40.78it/s]
